In [ ]:
import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()


Saving International_T20_Data.csv to International_T20_Data (1).csv


In [ ]:
df = pd.read_csv("International_T20_Data.csv")
df.head()

,innings,meta.data_version,meta.created,meta.revision,info.dates,info.gender,info.match_type,info.outcome.by.wickets,info.outcome.winner,info.overs,...,info.outcome.by.runs,info.match_type_number,info.neutral_venue,info.outcome.method,info.outcome.result,info.outcome.eliminator,info.supersubs.New Zealand,info.supersubs.South Africa,info.bowl_out,info.outcome.bowl_out
0,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-18,2,"[datetime.date(2017, 2, 17)]",male,T20,5.0,Sri Lanka,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-19,2,"[datetime.date(2017, 2, 19)]",male,T20,2.0,Sri Lanka,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-23,1,"[datetime.date(2017, 2, 22)]",male,T20,NaN,Australia,20,...,41.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"[{'1st innings': {'team': 'Hong Kong', 'delive...",0.9,2016-09-12,1,"[datetime.date(2016, 9, 5)]",male,T20,NaN,Hong Kong,20,...,40.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"[{'1st innings': {'team': 'Zimbabwe', 'deliver...",0.9,2016-06-19,1,"[datetime.date(2016, 6, 18)]",male,T20,NaN,Zimbabwe,20,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#1)Rename all the column names to appropriate names.(Example: meta.created → created_date)
df.columns = df.columns.str.replace('.', '_')
df.columns


Index(['innings', 'meta_data_version', 'meta_created', 'meta_revision',
       'info_dates', 'info_gender', 'info_match_type',
       'info_outcome_by_wickets', 'info_outcome_winner', 'info_overs',
       'info_player_of_match', 'info_teams', 'info_toss_decision',
       'info_toss_winner', 'info_umpires', 'info_venue', 'info_city',
       'info_outcome_by_runs', 'info_match_type_number', 'info_neutral_venue',
       'info_outcome_method', 'info_outcome_result', 'info_outcome_eliminator',
       'info_supersubs_New Zealand', 'info_supersubs_South Africa',
       'info_bowl_out', 'info_outcome_bowl_out'],
      dtype='object')

In [ ]:
#2)Find the top three venues which hosted the greatest number of matches.
top_venues = df['info_venue'].value_counts().head(3)
print("Top 3 Venues:")
print(top_venues)


Top 3 Venues:
info_venue
Dubai International Cricket Stadium    62
Sheikh Zayed Stadium                   41
Shere Bangla National Stadium          39
Name: count, dtype: int64


In [ ]:
#3)Find the pair of cricket teams who played the most number of matches against each other
import ast

df['info_teams'] = df['info_teams'].apply(ast.literal_eval)
df['team1'] = df['info_teams'].apply(lambda x: x[0])
df['team2'] = df['info_teams'].apply(lambda x: x[1])
df['team_pair'] = df.apply(lambda x: tuple(sorted([x['team1'], x['team2']])), axis=1)

most_matches_pair = df['team_pair'].value_counts().head(1)

print("Teams that played the most matches against each other:")
print(most_matches_pair)

Teams that played the most matches against each other:
team_pair
(Australia, England)    45
Name: count, dtype: int64


In [ ]:
 #4)Print the top five teams by their win percentages
matches_played = pd.concat([df['team1'], df['team2']]).value_counts()

matches_won = df['info_outcome_winner'].value_counts()

win_percentage = (matches_won / matches_played) * 100

top5_teams = win_percentage.sort_values(ascending=False).head(5)

print("Top 5 Teams by Win Percentage:")
print(top5_teams)

Top 5 Teams by Win Percentage:
Belgium        100.000000
Spain           83.333333
Germany         76.470588
Namibia         73.529412
Afghanistan     68.000000
Name: count, dtype: float64


In [ ]:
#5)Function to get the scorecard of each match
def get_scorecard(match_index):

    import pandas as pd

    match = df.iloc[match_index]
    innings_data = match['innings']

    scorecards = []

    for inn in innings_data:

        innings = list(inn.values())[0]
        team = innings['team']
        deliveries = innings['deliveries']

        balls = []

        # flatten deliveries
        for d in deliveries:
            ball = list(d.values())[0]
            runs = ball['runs']

            balls.append({
                'batsman': ball['batsman'],
                'bowler': ball['bowler'],
                'runs_batsman': runs['batsman'],
                'runs_total': runs['total']
            })

        df_innings = pd.DataFrame(balls)

        # top batsmen
        top_batsmen = (
            df_innings.groupby('batsman')['runs_batsman']
            .sum()
            .sort_values(ascending=False)
            .head(4)
        )

        # top bowlers
        top_bowlers = (
            df_innings.groupby('bowler')['runs_total']
            .sum()
            .sort_values()
            .head(4)
        )

        scorecard = pd.concat([top_batsmen, top_bowlers], axis=1)

        print("\nTeam:", team)
        print(scorecard)

        scorecards.append(scorecard)

    return scorecards


In [ ]:
get_scorecard(0)
